In [ ]:
%load_ext autoreload
%autoreload 2


# LLM → SPICE → GDS (Clocked Comparator)
SSCS Chipathon 2026 — gLayout Track

Describe your analog circuit in natural language.
An LLM (DeepSeek) generates the SPICE netlist,
then gLayout produces the DRC-clean GDSII layout.

**PDK**: gf180mcuD  |  **Supply**: 1.8 V  |  **LLM**: DeepSeek

In [ ]:
import sys, os
from pathlib import Path
root = Path("/foss/designs/chipathon2026-D")
sys.path.insert(0, str(root))

from glayout import gf180
from gdsfactory import Component
import gdstk, IPython.display

from core import (
    llm_to_gds, generate_netlist_from_prompt,
    spice_to_gds, display_component,
    run_comparator_tran, run_comparator_pvt,
    run_drc, run_lvs, run_pex,
)

In [ ]:
# ==========================================
# 0. Setup API Key
# ==========================================
# Option A: export DEEPSEEK_API_KEY=sk-...
# Option B: edit file .env (copy from .env.example)

# Verify key is loaded:
from core.pipeline import _load_api_key
key = _load_api_key()
if key:
    print(f"API key loaded: {key[:10]}...")
else:
    print("WARNING: DEEPSEEK_API_KEY not set!")
    print("  Run: export DEEPSEEK_API_KEY=sk-...")
    print("  Or:  cp .env.example .env && edit .env")

In [ ]:
# ==========================================
# 1. Describe Your Circuit (Natural Language)
# ==========================================

prompt = """
Design a clocked StrongARM latch comparator.
Use gf180mcuD PDK with 1.8V supply.
The circuit should have:
- NMOS tail transistor controlled by clk
- NMOS differential input pair (W=8u L=0.5u)
- PMOS cross-coupled pair (W=4u L=0.5u)
- NMOS cross-coupled pair (W=3u L=0.5u)
- PMOS reset switches controlled by clk (W=2u L=0.5u)
- Output inverters (PMOS W=4u, NMOS W=2u, L=0.5u)
Subcircuit name: my_comp
Ports: vin_p, vin_n, clk, vout_p, vout_n, vdd, vss

this is the example of StrongArm latch comparator, you can use it as a reference for your design:

.lib "/home/huda/.volare/gf180mcuD/libs.tech/ngspice/sm141064.ngspice" typical
.subckt comp_strongarm vin_p vin_n clk vout_p vout_n vdd vss
Mtail  n1 clk vss vss nfet_03v3 W=5u L=0.5u
M2 n2 vin_p n1 vss nfet_03v3 W=8u L=0.5u
M3 n3 vin_n n1 vss nfet_03v3 W=8u L=0.5u
M4 n2 n3 vdd vdd pfet_03v3 W=4u L=0.5u
M5 n3 n2 vdd vdd pfet_03v3 W=4u L=0.5u
M6 n2 n3 n1 vss nfet_03v3 W=3u L=0.5u
M7 n3 n2 n1 vss nfet_03v3 W=3u L=0.5u
M8 n2 clk vdd vdd pfet_03v3 W=2u L=0.5u
M9 n3 clk vdd vdd pfet_03v3 W=2u L=0.5u
M10 vout_p n2 vdd vdd pfet_03v3 W=4u L=0.5u
M11 vout_p n2 vss vss nfet_03v3 W=2u L=0.5u
M12 vout_n n3 vdd vdd pfet_03v3 W=4u L=0.5u
M13 vout_n n3 vss vss nfet_03v3 W=2u L=0.5u
.ends

Please generate the SPICE netlist for this comparator design based on the specifications provided above and make any necessary adjustments and improvements not copy and paste the example.

"""

In [ ]:
# ==========================================
# 2. LLM → SPICE Netlist
# ==========================================

netlist = generate_netlist_from_prompt(prompt)
if netlist:
    print(netlist)
else:
    print("LLM failed to generate netlist. Check your API key.")

In [ ]:
# ==========================================
# 3. Simulation: Delay, Offset & PVT
# ==========================================
# Uses ngspice for transient analysis.

cell_name = "my_comp"

pre = run_comparator_tran(netlist, cell_name,
                           vdd=1.8, vcm=0.9,
                           clk_period=1e-6, tstop=5e-6)

off = run_comparator_tran(netlist, cell_name, 
                           vdd=1.8, vcm=0.9, 
                           clk_period=1e-6, tstop=7e-6, 
                           measure_offset=True)

pvt = run_comparator_pvt(netlist, cell_name, 
                           vdd=1.8, vcm=0.9, 
                           clk_period=1e-6, tstop=5e-6)

finetune_needed = pre.get('finetune', True) or off.get('finetune', True) or pvt.get('finetune', True)

print("--- DELAY FEEDBACK ---")
print(pre.get("llm_feedback"))
print("\n--- OFFSET FEEDBACK ---")
print(off.get("llm_feedback"))
print("\n--- PVT FEEDBACK ---")
print(pvt.get("llm_feedback"))
print(f"\nOVERALL FINETUNE NEEDED: {finetune_needed}")


In [ ]:
# ==========================================
# 4. LLM Finetuning
# ==========================================

# Use the combined finetune_needed from the previous cell
while finetune_needed:
    # Combine all feedback
    combined_feedback = pre.get("llm_feedback", "") + "\n" + off.get("llm_feedback", "") + "\n" + pvt.get("llm_feedback", "")
    print("Sending feedback to LLM... Please wait.")
    
    # Generate new netlist
    netlist = generate_netlist_from_prompt(prompt, llm_feedback=combined_feedback)
    
    # Rerun all 3 simulations
    pre = run_comparator_tran(netlist, cell_name, vdd=1.8, vcm=0.9, clk_period=1e-6, tstop=5e-6)
    off = run_comparator_tran(netlist, cell_name, vdd=1.8, vcm=0.9, clk_period=1e-6, tstop=7e-6, measure_offset=True)
    pvt = run_comparator_pvt(netlist, cell_name, vdd=1.8, vcm=0.9, clk_period=1e-6, tstop=5e-6)
    
    # Re-evaluate
    finetune_needed = pre.get('finetune', True) or off.get('finetune', True) or pvt.get('finetune', True)
    
    if finetune_needed:
        print("Still needs finetuning. Iterating again...")
    else:
        print("Success! The comparator is fully functional across all PVT corners.")


In [ ]:
# ==========================================
# 4. SPICE → GDS Layout
# ==========================================

if netlist:
    GDS_PATH = os.path.join(os.getcwd(), "llm_out.gds")
    result = spice_to_gds(netlist, mode="analog", add_labels=True)
    result.write_gds(GDS_PATH)
    print(f"Layout written to {GDS_PATH}")
    display_component(result, scale=0.5)